In [ ]:
# ============================================================
# 0. Importing libraries and modules
# ============================================================
import pandas as pd
import numpy as np
import scipy.io as sci
import matplotlib.pyplot as plt
import urllib.request
import io
import os
from scipy.stats import kurtosis
from scipy.signal import welch              # Analyse spectral Welch
from scipy.fftpack import fft, fftfreq      # Analyse spectral FFT

# ============================================================
# 1. Chargement des données 
# ============================================================

    # ==================
    # 1.1 Contexte
    # ==================

# --- Fréquences caractéristiques des roulements --- #
# Source : https://engineering.case.edu/bearingdatacenter/bearing-information
COEFF_Freq_DE = {"BPFI": 5.4152, "BPFO": 3.5848, "FTF": 0.39828, "BSF": 4.7135} 
COEFF_Freq_FE = {"BPFI": 4.9469, "BPFO": 3.0530, "FTF": 0.3817, "BSF": 3.9874}
# BPFI (Ball Pass Frequency Inner)      >> Piste intérieur
# BPFO (Ball Pass Frequency Outer)      >> Piste extérieur
# BSF (Ball Spin Frequency)             >> Bille
# FTF (Fundamental Train Frequency)     >> Cage

FS = 12000  # fréquence d'échantillonnage (Hz)

    # ==================
    # 1.2 Data
    # ==================
# Local data path
LocalDataPath = "../../data/CWRU_data/"

# Data loading
def charger_signaux(chemin_mat):
    # Si c'est un dossier, prend le premier fichier .mat
    if os.path.isdir(LocalDataPath + chemin_mat):
        fichier = next(f for f in os.listdir(LocalDataPath + chemin_mat) if f.endswith('.mat'))
        chemin_mat = chemin_mat.rstrip('/') + '/' + fichier

    mat = sci.loadmat(LocalDataPath+chemin_mat)
    signaux = {}
    for cle, valeur in mat.items():
        if cle.startswith("__"):
            continue
        if cle.endswith("_DE_time"):
            signaux["DE"] = np.ravel(valeur)
        elif cle.endswith("_FE_time"):
            signaux["FE"] = np.ravel(valeur)
        elif cle.endswith("_BA_time"):
            signaux["BA"] = np.ravel(valeur)
        elif cle.endswith("RPM"):
            signaux["RPM"] = float(valeur[0][0])
    return signaux

# ============================================================
# 2. Chargement des données
# ============================================================
normal = charger_signaux('Normal_Baseline_Data/Motor_Load_1/Normal_1')
fault = charger_signaux('12k_Fan_End_Bearing_Fault/Fault_Diam_007/Motor_Load_1/IR007_1')
# fault = charger_signaux('12k_Fan_End_Bearing_Fault/Fault_Diam_007/Motor_Load_1/B007_1')
# fault = charger_signaux('12k_Fan_End_Bearing_Fault/Fault_Diam_007/Motor_Load_1/OR007@6_1')


# --- Dictionnaire de signaux
signaux = {
    "Normal Drive": normal.get("DE"),
    "Normal Fan":   normal.get("FE"),
    "Fault Drive":  fault.get("DE"),
    "Fault Fan":    fault.get("FE"),
    "Fault Base":   fault.get("BA"),
    "RPM":          fault.get("RPM"),
}
# supprime les vides
signaux = {nom: sig for nom, sig in signaux.items() if sig is not None}
rpm=signaux["RPM"]

# ============================================================
# 2. Définitions des fontions
# ============================================================

# --- Calcul de indicateurs de base d'un signal --- #
def indicateurs(signal):
    return {
        "N points":  len(signal),
        "Durée (sec)":  len(signal)/FS,
        "Moyenne":   np.mean(signal),
        "Écart-type": np.std(signal),
        "RMS":       np.sqrt(np.mean(signal**2)),
        "Crête":     np.max(np.abs(signal)),
        "Facteur crête": np.max(np.abs(signal)) / np.sqrt(np.mean(signal**2)),
        "Kurtosis":  kurtosis(signal, fisher=False)
    }


# ============================================================
# 3. Affichage graphiques
# ============================================================

# fenetre 0,1 secondes
fig, axes = plt.subplots(3, 2, figsize=(14, 8), sharey=True)
axes[0,0].plot(signaux["Normal Drive"][:1200], color="g", lw=0.5); axes[0,0].set_title("Normal Drive")
axes[1,0].plot(signaux["Normal Fan"][:1200], color="g", lw=0.5); axes[1,0].set_title("Normal Fan")
axes[0,1].plot(signaux["Fault Drive"][:1200], color="r", lw=0.5); axes[0,1].set_title("Fault Drive")
axes[1,1].plot(signaux["Fault Fan"][:1200], color="r", lw=0.5); axes[1,1].set_title("Fault Fan")
axes[2,1].plot(signaux["Fault Base"][:1200], color="r", lw=0.5); axes[2,1].set_title("Fault Base")
plt.tight_layout()
fig.suptitle(f"Défaut ventilateur -  piste intérieure  -  charge motor 1  -  défault diam 0.007\"  -  sampling 12K  -  rpm {rpm}", fontsize=20, y=1.03)
for ax in axes[:, 0]: ax.set_ylabel("Amplitude")
for ax in axes[-1, :]: ax.set_xlabel("Temps 1/12000 sec")
plt.show()


# ============================================================
# 4. Indicateurs
# ============================================================

tableau = pd.DataFrame({"Normal_Drive": indicateurs(signaux["Normal Drive"]),
                        "Normal_Fan": indicateurs(signaux["Normal Fan"]),
                        "Fault_Drive": indicateurs(signaux["Normal Drive"]),
                        "Fault_Fan": indicateurs(signaux["Fault Fan"]),
                        "Fault_Base": indicateurs(signaux["Fault Base"]),                        
                        }).T.round(2)
display(tableau)

# print("Le Kurtosis du signal Fault_Fan a le plus de différence avec le signal normal (2.87 > 6.10),\nce qui indique un défaut sur ce roulement plutôt que sur le Fault_Drive (2.93 > 3.91)")

FileNotFoundError: [Errno 2] No such file or directory: '/home/ubuntu/data/Normal_Baseline_Data/Motor_Load_1/Normal_1.mat'